# 09 — Estudo de sementes do RF no teste (padrão × ajustado)

**Pergunta:** a diferença que vimos no teste entre o RF padrão (100 árvores, sem limite de profundidade, folha 1) e o ajustado (300 árvores, profundidade 15, folha 1) é maior que a variação causada só pelo **sorteio do modelo** (a semente)?

O Random Forest sorteia as linhas de cada árvore e as colunas de cada divisão. Com outra semente, o modelo sai um pouco diferente e a nota no teste muda um pouco. Até aqui cada modelo foi treinado com **uma** semente (42). Este notebook treina cada um com várias e mostra a **faixa** de resultados.

| O que | Como |
|---|---|
| **Modelos** | Os mesmos dois hiperparâmetros de antes, com **5 sementes** cada (`SEMENTES_ESTUDO`, padrão 42 a 46). A semente 42 usa os **modelos já gravados** (notebooks 05 e 07); as outras são treinadas agora |
| **Dados de treino** | O mesmo refit do notebook 05 (101.536 linhas), lido na mesma ordem do notebook 07. **Só a semente muda** |
| **Teste** | Lido **uma vez** (junção com o tipo de ataque, em lotes, com checkpoint), e todos os modelos preveem cada lote |
| **Métricas** | ROC-AUC e PR-AUC do BENIGN (não dependem de limiar), e no limiar 0,5 e no limiar de 0,1% da validação de cada modelo: recall, falso alarme, MCC, ataques perdidos, alarmes falsos e o recall do tipo `Syn` |
| **Comparação** | Pareada por semente (ajustado − padrão): média, desvio, mínimo, máximo e em quantas sementes o ajustado foi melhor |

## O que este estudo é e não é
- **É descritivo.** Mede o ruído do sorteio do modelo e dá uma faixa aos resultados. **Nenhuma escolha depende dele**: nem hiperparâmetros, nem limiares, nem semente.
- **Usa o teste de novo**, só para descrever. Isso deve constar no texto do TCC.
- **Não mede** a variação por usar outras linhas no ajuste ou no refit (os dados de treino são os mesmos).

## O que o notebook grava
Na pasta `estudo_sementes` dentro da execução do notebook 07: `resultados_por_semente.csv`, `resumo_por_modelo.csv`, `diferenca_pareada.csv`, `estudo_sementes.json`, `grafico_sementes.png`, os modelos novos (`modelos/`) e os lotes (`lotes_*`). **Nada do notebook 08 é alterado.**

## Antes de rodar
Os notebooks 05, 07 e 08 precisam ter terminado. Feche os outros notebooks (cada kernel guarda memória). **Não mude `LOTE_LINHAS` no meio de uma execução.** Se for interrompido, rode a célula de novo: os modelos e os lotes prontos são reaproveitados. Tempo estimado: ~12 a 15 minutos (leitura do teste ~4 min, previsão dos 10 modelos ~6 min, treino dos 8 modelos novos ~2 min).

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


### Ferramentas do teste
- **`ler_csv_com_espera`**: lê o teste em CSV direto para a tabela (~200 MiB por 250 mil linhas, em vez de ~750 MiB pelo JSON), traz a coluna de **texto** com o tipo de ataque e, se o servidor ficar sem memória (`Code 241`), espera e repete.
- **`Barra`**: barra de progresso com tempo restante estimado.
- **`recall_por_tipo`**: recall de cada tipo de ataque (na primeira posição, a especificidade do BENIGN).

In [6]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

Autoteste das ferramentas do teste OK


### Métricas leves em memória
Com ~7,2 milhões de previsões, as funções do sklearn ordenam cópias dos dados várias vezes (**~650 MiB** a mais para 3 limiares, no meu teste) e podem dar `MemoryError`. As métricas aqui saem de **contagens de ataques e BENIGN por valor de escore** (um RF de 100 árvores tem só ~100 valores): o resultado é **idêntico** ao do sklearn (diferença máxima de 2×10⁻¹⁶ nos meus testes) e gasta quase nenhuma memória.

In [7]:
_trap = getattr(np, "trapezoid", None) or getattr(np, "trapz")

def contagens_por_escore(y, p, bloco=1_000_000):
    # Conta ataques (y = 1) e BENIGN (y = 0) por VALOR DISTINTO do escore. É tudo o que as métricas precisam e ocupa memória mínima
    # (as funções do sklearn ordenam cópias de 7 milhões de elementos várias vezes). Os escores são comparados em float32, como foram gravados.
    y = np.asarray(y); p = np.asarray(p, dtype=np.float32)
    vals = np.unique(p)
    if len(vals) > 200_000:                                            # escores quase contínuos: caminho de uma passada
        vals, inv = np.unique(p, return_inverse=True)
        n_tot = np.bincount(inv, minlength=len(vals)); n_pos = np.bincount(inv, weights=y, minlength=len(vals))
    else:
        n_tot = np.zeros(len(vals), np.int64); n_pos = np.zeros(len(vals), np.float64)
        for i in range(0, len(p), bloco):
            idx = np.searchsorted(vals, p[i:i + bloco]); n_tot += np.bincount(idx, minlength=len(vals)); n_pos += np.bincount(idx, weights=y[i:i + bloco], minlength=len(vals))
    n_pos = np.rint(n_pos).astype(np.int64); n_tot = n_tot.astype(np.int64)
    return vals, n_pos, n_tot - n_pos

def _curva(tp_c, fp_c, P, N):
    tpr = np.concatenate(([0.0], tp_c / P)); fpr = np.concatenate(([0.0], fp_c / N)); rec = tp_c / P; prec = tp_c / (tp_c + fp_c)
    return fpr, tpr, float(_trap(tpr, fpr)), float(np.sum(np.diff(np.concatenate(([0.0], rec))) * prec))

def metricas_hist(vals, n_pos, n_neg, limiar=0.5):
    # Mesmas métricas de `metricas` (sem pesos), a partir das contagens por escore. Previsão = escore >= limiar (ambos em float32).
    m = vals >= np.float32(limiar)
    tp, fp, fn, tn = float(n_pos[m].sum()), float(n_neg[m].sum()), float(n_pos[~m].sum()), float(n_neg[~m].sum())
    P, N = tp + fn, tn + fp
    rec, esp, prec = _div(tp, P), _div(tn, N), _div(tp, tp + fp); prec_b = _div(tn, tn + fn)
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)); d = slice(None, None, -1)
    _, _, roc_auc, pr_a = _curva(np.cumsum(n_pos[d]), np.cumsum(n_neg[d]), P, N)       # ataque como positivo (escores em ordem decrescente)
    _, _, _, pr_b = _curva(np.cumsum(n_neg), np.cumsum(n_pos), N, P)                    # BENIGN como positivo (escore 1 - p: ordem crescente de p)
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den), "precisao_benign": prec_b, "f1_benign": f1_b,
            "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": pr_b, "roc_auc": roc_auc, "pr_auc": pr_a, "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr_hist(vals, n_pos, n_neg, alvo):
    # O MAIOR recall com taxa de falso alarme <= alvo, lido na curva ROC exata (todos os pontos distintos).
    d = slice(None, None, -1); P, N = n_pos.sum(), n_neg.sum()
    tpr = np.concatenate(([0.0], np.cumsum(n_pos[d]) / P)); fpr = np.concatenate(([0.0], np.cumsum(n_neg[d]) / N)); thr = np.concatenate(([np.inf], vals[d].astype(float)))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0); lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

## 1. Carregar os modelos originais e o teste
Carrega o modelo padrão (notebook 05) e o ajustado (notebook 07), com a semente original, e descobre os tipos de ataque do teste.

In [8]:
# ---------- 1.1 Parâmetros e carga ----------
import hashlib
PASTA_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_binario_busca"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("rf_busca_*") if (p / "modelo_final_refit.joblib").exists() and (p / "FINALIZADO").exists())
    assert runs, "nenhuma pasta rf_busca_* concluída com modelo_final_refit.joblib em " + str(PASTA_BASE) + ": rode antes o notebook 07"
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN)
PASTA_PADRAO = globals().get("PASTA_PADRAO", None)
if PASTA_PADRAO is None:
    base_pad = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_binario"
    rp = sorted(p for p in base_pad.glob("rf_bin_*") if (p / "modelo_final_refit.joblib").exists()); assert rp, "nenhum modelo padrão (notebook 05) encontrado em " + str(base_pad)
    PASTA_PADRAO = rp[-1]
PASTA_PADRAO = Path(PASTA_PADRAO)
SEMENTES = tuple(globals().get("SEMENTES_ESTUDO", (42, 43, 44, 45, 46)))
LOTE_LINHAS = globals().get("LOTE_LINHAS", 50_000)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
CONFERIR_REPRODUCAO = globals().get("CONFERIR_REPRODUCAO", True)
VIEW_TESTE, TAB_TESTE, TAB_TREINO, V_REFIT = "prep_vw_rf_bin_teste_sem_sobrep", "prep_final_teste", "prep_final_treino", "prep_vw_bal_bin_refit"
assert existe(VIEW_TESTE) and existe(V_REFIT), "faltam as views do teste ou do refit: rode os notebooks 03 a 05"

def carregar(pasta):
    mdl = joblib.load(pasta / "modelo_final_refit.joblib"); pc = json.loads((pasta / "preprocessamento.json").read_text(encoding="utf-8"))
    cf = json.loads((pasta / "config_treino.json").read_text(encoding="utf-8")); lv = json.loads((pasta / "limiares_validacao.json").read_text(encoding="utf-8"))
    pp = PreprocessadorRF(pc["features"], pc["medianas_treino"], pc["colunas_retiradas_ablacao"], pc["versao_views_rf"])
    assert pp.colunas_modelo == pc["colunas_modelo"] == list(mdl.colunas_modelo_), "a ordem das colunas do modelo difere da gravada"
    assert all(getattr(mdl, k) == v for k, v in cf["hiperparametros"].items()), "o modelo carregado não tem os hiperparâmetros gravados"
    return mdl, pp, cf, lv
modelo_aj, prep, config_aj, lim_aj = carregar(PASTA_RUN); modelo_pad, prep_pad, config_pad, lim_pad = carregar(PASTA_PADRAO); CM = prep.colunas_modelo
assert prep_pad.colunas_modelo == CM, "o modelo padrão usa colunas diferentes do ajustado"
cols_view = colunas_view(VIEW_TESTE); assert cols_view[:2] == ["id_linha", "y"] and cols_view[2:] == CM, "as colunas da view de teste não coincidem com as dos modelos"
SEMENTE_ORIG = int(config_aj["hiperparametros"]["random_state"]); assert SEMENTE_ORIG == int(config_pad["hiperparametros"]["random_state"]), "os dois modelos originais usam sementes diferentes"
assert SEMENTE_ORIG in SEMENTES, f"a semente original ({SEMENTE_ORIG}) precisa estar em SEMENTES_ESTUDO"
BASE = {"padrão": (modelo_pad, config_pad["hiperparametros"], lim_pad), "ajustado": (modelo_aj, config_aj["hiperparametros"], lim_aj)}
LIM_01 = {n: float(b[2]["pontos_de_operacao_validacao"][1]["limiar"]) for n, b in BASE.items()}          # limiar de 0,1% de falso alarme, gravado na validação de cada modelo original
N_TESTE = n_linhas(VIEW_TESTE)

nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
tt = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {TAB_TESTE} WHERE sobrepoe_treino = 0 GROUP BY label_multi"); tt["tipo"] = tt["tipo"].map(nome_tipo)
TIPOS = ["BENIGN"] + sorted(t for t in set(tt["tipo"]) if t != "BENIGN"); COD = {t: i for i, t in enumerate(TIPOS)}; NT = len(TIPOS)
PASTA_EST = PASTA_RUN / "estudo_sementes"; (PASTA_EST / "modelos").mkdir(parents=True, exist_ok=True)
print("⚠ Este estudo usa o TESTE de novo, só para descrever a variação por semente. Nenhuma escolha depende dele.")
print(f"Modelo padrão: {PASTA_PADRAO.name} | { {k: BASE['padrão'][1][k] for k in ('n_estimators', 'max_depth', 'min_samples_leaf')} }")
print(f"Modelo ajustado: {PASTA_RUN.name} | { {k: BASE['ajustado'][1][k] for k in ('n_estimators', 'max_depth', 'min_samples_leaf')} }")
print(f"Sementes: {SEMENTES} (a original, {SEMENTE_ORIG}, usa os modelos já gravados) | teste: {N_TESTE:,} linhas | tipos: {NT - 1} ataques + BENIGN")
print(f"Pasta do estudo: {PASTA_EST}")

⚠ Este estudo usa o TESTE de novo, só para descrever a variação por semente. Nenhuma escolha depende dele.
Modelo padrão: rf_bin_20261021_221311 | {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1}
Modelo ajustado: rf_busca_20261024_175105 | {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 1}
Sementes: (42, 43, 44, 45, 46) (a original, 42, usa os modelos já gravados) | teste: 7,219,189 linhas | tipos: 7 ataques + BENIGN
Pasta do estudo: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_busca\rf_busca_20261024_175105\estudo_sementes


## 2. Treino das sementes novas
Lê o **mesmo refit** do notebook 05 (ordenado por `id_linha`, como no notebook 07) e treina, para cada modelo, as sementes que faltam. Cada modelo novo é gravado em `modelos/` (se a execução for interrompida, os já prontos são reaproveitados). Uma conferência retreina o modelo ajustado com a semente original e confirma que sai **idêntico** ao do notebook 07 (isso mostra que os dados e o treino são os mesmos; no modelo padrão, o notebook 05 não ordenava as linhas, então essa conferência não se aplica).

In [9]:
# ---------- 2.1 Treino das sementes novas ----------
def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
rf_df = ordenar(ler_view(V_REFIT)); assert colunas_view(V_REFIT)[:2] == ["id_linha", "y"] and colunas_view(V_REFIT)[2:] == CM, "as colunas do refit não coincidem com as do modelo"
X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int); assert np.isfinite(X_rf).all(), "valores não finitos no refit"
print(f"Refit: {len(X_rf):,} linhas ({int((y_rf == 0).sum()):,} BENIGN, {int((y_rf == 1).sum()):,} ataques)")
MODELOS = {(nome, SEMENTE_ORIG): BASE[nome][0] for nome in BASE}; tempos = []
novas = [(nome, s) for nome in BASE for s in SEMENTES if s != SEMENTE_ORIG]
barra = Barra(len(novas) + (1 if CONFERIR_REPRODUCAO else 0), "Treino das sementes novas")
for nome, s in novas:
    hp = BASE[nome][1]; arq = PASTA_EST / "modelos" / f"rf_{nome}_semente{s}.joblib"
    if arq.exists():
        m = joblib.load(arq); assert m.random_state == s and all(getattr(m, k) == v for k, v in hp.items() if k != "random_state"), f"o modelo gravado {arq.name} não tem os hiperparâmetros esperados"; dt = float("nan"); txt = "já pronto"
    else:
        t0 = time.perf_counter(); m = RandomForestClassifier(**{**hp, "random_state": s}).fit(X_rf, y_rf); dt = time.perf_counter() - t0; m.colunas_modelo_ = list(CM)
        joblib.dump(m, arq, compress=COMPRESSAO_MODELO); txt = f"treinado em {dt:.0f} s"
    MODELOS[(nome, s)] = m; tempos.append({"modelo": nome, "semente": s, "relogio_s": dt}); barra.avancar(1, f"{nome}, semente {s}: {txt}")
REPRODUZ = None
if CONFERIR_REPRODUCAO:
    hp = BASE["ajustado"][1]; m_chk = RandomForestClassifier(**hp).fit(X_rf, y_rf)
    REPRODUZ = bool(np.allclose(m_chk.predict_proba(X_rf[:5000])[:, 1], BASE["ajustado"][0].predict_proba(X_rf[:5000])[:, 1])); del m_chk
    barra.avancar(1, f"conferência: ajustado com a semente {SEMENTE_ORIG} {'reproduz' if REPRODUZ else 'NÃO reproduz'} o modelo do notebook 07")
barra.fechar()
pd.DataFrame(tempos).to_csv(PASTA_EST / "tempos_treino_sementes.csv", index=False)
CHAVES = [(nome, s) for nome in BASE for s in SEMENTES]
print(f"✔ {len(CHAVES)} modelos prontos ({len(novas)} novos + 2 originais)" + ("" if REPRODUZ is None else f" | o ajustado retreinado com a semente original {'é IDÊNTICO ao do notebook 07' if REPRODUZ else 'DIFERE do notebook 07: confira se o refit/ordem dos dados mudou'}"))

Refit: 101,536 linhas (50,768 BENIGN, 50,768 ataques)


Treino das sementes novas: concluído em 2min 06s
✔ 10 modelos prontos (8 novos + 2 originais) | o ajustado retreinado com a semente original é IDÊNTICO ao do notebook 07


## 3. Previsão no teste (uma única leitura)
Lê o teste em lotes (faixas de `id_linha`, com o tipo de ataque de cada linha) e **todos os modelos preveem cada lote**. Cada lote é gravado em `lotes_<código>/`; se a execução for interrompida, rode a célula de novo.

In [10]:
# ---------- 3.1 Previsão em lotes ----------
IMPRESSAO = hashlib.sha1(json.dumps([[f"{n}:{s}" for n, s in CHAVES], LOTE_LINHAS, N_TESTE]).encode()).hexdigest()[:8]
B = max(1, math.ceil(N_TESTE / LOTE_LINHAS)); PASTA_LOTES = PASTA_EST / f"lotes_{IMPRESSAO}"; PASTA_LOTES.mkdir(exist_ok=True)
cols_sql = ", ".join("v." + q(c) for c in CM); ys, gs, Ps, Ts, n_lote = [], [], [], [], []; reaproveitados = 0
barra = Barra(B, "Previsão no teste"); t_inicio = time.perf_counter()
for j in range(B):
    arq = PASTA_LOTES / f"lote_{j:03d}_de_{B:03d}.npz"
    if arq.exists():
        d = np.load(arq); ys.append(d["y"]); gs.append(d["g"]); Ps.append(d["P"]); Ts.append(d["T"]); n_lote.append(len(d["y"])); reaproveitados += 1; barra.avancar(1, f"lote {j + 1}/{B}: já pronto"); continue
    lo, hi = j * 2 ** 64 // B, (j + 1) * 2 ** 64 // B
    onde_v = f"v.id_linha >= {lo}" + (f" AND v.id_linha < {hi}" if j < B - 1 else ""); onde_t = f"id_linha >= {lo}" + (f" AND id_linha < {hi}" if j < B - 1 else "")
    barra.avancar(0, f"lote {j + 1}/{B}: lendo... (memória do processo {rss_mib():,.0f} MiB)")
    df = ler_csv_com_espera(f"SELECT v.y, {cols_sql}, t.tipo FROM {VIEW_TESTE} AS v INNER JOIN (SELECT id_linha, label_multi AS tipo FROM {TAB_TESTE} WHERE {onde_t}) AS t ON v.id_linha = t.id_linha WHERE {onde_v}",
                            ["y"] + CM + ["tipo"], texto=("tipo",))
    if len(df) == 0:
        z8, zP, zT = np.zeros(0, np.int8), np.zeros((len(CHAVES), 0), np.float32), np.zeros(len(CHAVES)); np.savez(arq, y=z8, g=z8, P=zP, T=zT)
        ys.append(z8); gs.append(z8); Ps.append(zP); Ts.append(zT); n_lote.append(0); barra.avancar(1, f"lote {j + 1}/{B}: faixa sem linhas"); continue
    X, y = df[CM].to_numpy(float), df["y"].to_numpy(np.int8); assert np.isfinite(X).all(), "valores não finitos no teste"
    g = df["tipo"].map(nome_tipo).map(COD); assert g.notna().all(), "tipo de ataque do teste não reconhecido"; g = g.to_numpy(np.int8)
    assert ((g == 0) == (y == 0)).all(), "o tipo BENIGN não coincide com y = 0 no teste"
    P, T = np.zeros((len(CHAVES), len(y)), np.float32), np.zeros(len(CHAVES))
    for i, k in enumerate(CHAVES):
        t0 = time.perf_counter(); P[i] = MODELOS[k].predict_proba(X)[:, 1]; T[i] = time.perf_counter() - t0
    np.savez(arq, y=y, g=g, P=P, T=T); ys.append(y); gs.append(g); Ps.append(P); Ts.append(T); n_lote.append(len(y))
    barra.avancar(1, f"lote {j + 1}/{B}: {len(y):,} linhas, {len(CHAVES)} modelos"); del df, X; gc.collect()
barra.fechar()
if reaproveitados: print(f"  {reaproveitados} lote(s) já estavam prontos e foram reaproveitados")
Y, G = np.concatenate(ys), np.concatenate(gs); T_TOTAL = np.sum(Ts, axis=0); del ys, gs
assert len(Y) == N_TESTE, f"as linhas dos lotes ({len(Y):,}) não fecham com o total da view ({N_TESTE:,})"
PS = [np.concatenate([b[i] for b in Ps]) for i in range(len(CHAVES))]; del Ps; gc.collect()              # um vetor float32 por modelo
print(f"\n✔ {len(Y):,} previsões ({int((Y == 0).sum()):,} BENIGN = {100 * (Y == 0).mean():.3f}%) por {len(CHAVES)} modelos em {time.perf_counter() - t_inicio:.0f} s")

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.50 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.82 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 2.06 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.61 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 2.14 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.63 GiB); esperando até 30s e tentando de novo (1/5)...
  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 2.31 GiB); esperando até 30s e tentando de novo (1/5)...
Previsão no teste: concluído em 9m

## 4. Resultados por semente
Para cada modelo (padrão e ajustado) e cada semente: ROC-AUC e PR-AUC do BENIGN, e no limiar 0,5 e no limiar de 0,1% da validação **de cada modelo original** (fixo, não escolhido aqui): recall, falso alarme, MCC, ataques perdidos, alarmes falsos e o recall do tipo `Syn` (o que concentra as falhas). Depois, o resumo por modelo (média, desvio, mínimo e máximo) e a comparação **pareada** por semente (ajustado − padrão). Com 5 sementes, trate desvios e contagens como **indicativos**, não como testes estatísticos.

In [11]:
# ---------- 4.1 Resultados por semente ----------
ISYN = COD.get("Syn"); linhas = []
for i, (nome, s) in enumerate(CHAVES):
    p = PS[i]; VALS, NP, NN = contagens_por_escore(Y, p); m5 = metricas_hist(VALS, NP, NN, 0.5); ml = metricas_hist(VALS, NP, NN, LIM_01[nome]); rt = recall_por_tipo(Y, p, G, 0.5, NT)
    linhas.append({"modelo": nome, "semente": s, "original": s == SEMENTE_ORIG, "roc_auc": m5["roc_auc"], "pr_auc_benign": m5["pr_auc_benign"], "recall_05": m5["recall"], "fpr_05": m5["taxa_falso_alarme"], "mcc_05": m5["mcc"],
                   "fn_05": m5["fn"], "fp_05": m5["fp"], "recall_syn_05": float(rt[ISYN]) if ISYN is not None else float("nan"), "recall_fpr1pct_teste": recall_em_fpr_hist(VALS, NP, NN, 0.01)["recall"],
                   "limiar_0_1pct_validacao": LIM_01[nome], "recall_lim": ml["recall"], "fpr_lim": ml["taxa_falso_alarme"], "mcc_lim": ml["mcc"], "fn_lim": ml["fn"], "fp_lim": ml["fp"], "tempo_previsao_s": float(T_TOTAL[i])})
df_res = pd.DataFrame(linhas)
print("== Resultados no teste, por modelo e semente (original = a semente das etapas anteriores) ==")
display(df_res.round(5))

METS = ["roc_auc", "pr_auc_benign", "recall_05", "fpr_05", "mcc_05", "fn_05", "fp_05", "recall_syn_05", "recall_fpr1pct_teste", "mcc_lim", "fn_lim", "fp_lim"]
resumo_tab = pd.concat({n: df_res[df_res.modelo == n][METS].agg(["mean", "std", "min", "max"]).T for n in BASE}, axis=1)
print("\n== Resumo por modelo (média, desvio entre sementes, mínimo e máximo) =="); display(resumo_tab.round(5))

piv = lambda m: df_res.pivot(index="semente", columns="modelo", values=m)
BOM = {"roc_auc": "maior", "pr_auc_benign": "maior", "recall_05": "maior", "fpr_05": "menor", "mcc_05": "maior", "fn_05": "menor", "fp_05": "menor", "recall_syn_05": "maior", "recall_fpr1pct_teste": "maior", "mcc_lim": "maior", "fn_lim": "menor", "fp_lim": "menor"}
lin_par = []
for m in METS:
    d = piv(m)["ajustado"] - piv(m)["padrão"]; melhor = int((d > 0).sum() if BOM[m] == "maior" else (d < 0).sum())
    lin_par.append({"métrica": m, "melhor é": BOM[m], "diferença média (ajustado − padrão)": d.mean(), "desvio das diferenças": d.std(ddof=1), "mínima": d.min(), "máxima": d.max(), "ajustado melhor em": f"{melhor} de {len(d)}", "na semente original": d.loc[SEMENTE_ORIG]})
tab_par = pd.DataFrame(lin_par); print("\n== Diferença pareada por semente (ajustado − padrão) =="); display(tab_par.round(5))

a, b = piv("roc_auc")["ajustado"], piv("roc_auc")["padrão"]; d = a - b
print(f"\nROC-AUC no teste, {len(SEMENTES)} sementes: padrão {b.min():.5f} a {b.max():.5f} (média {b.mean():.5f}, desvio {b.std(ddof=1):.5f}) | ajustado {a.min():.5f} a {a.max():.5f} (média {a.mean():.5f}, desvio {a.std(ddof=1):.5f})")
print("→ as faixas NÃO se tocam: o ajustado ficou acima do padrão em todas as sementes." if a.min() > b.max() else "→ as faixas SE SOBREPÕEM: em alguma semente o padrão superou o pior ajustado (a diferença não é maior que o ruído do sorteio).")
print(f"Diferença pareada: média {d.mean():+.5f} (de {d.min():+.5f} a {d.max():+.5f}); na semente original ({SEMENTE_ORIG}), a das etapas anteriores: {d.loc[SEMENTE_ORIG]:+.5f}; o ajustado foi melhor em {int((d > 0).sum())} de {len(d)} sementes.")
if ISYN is not None:
    sa, sb = piv("recall_syn_05")["ajustado"], piv("recall_syn_05")["padrão"]; print(f"Recall do `Syn` no limiar 0,5: padrão {sb.min():.4f} a {sb.max():.4f} | ajustado {sa.min():.4f} a {sa.max():.4f}")

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    pain = [("roc_auc", "ROC-AUC no teste"), ("recall_syn_05", "recall do Syn (limiar 0,5)"), ("fpr_05", "falso alarme (limiar 0,5)")]; fig, ax = plt.subplots(1, 3, figsize=(12.5, 4)); rng = np.random.default_rng(0)
    for a_, (m, tit) in zip(ax, pain):
        for x, nome in enumerate(BASE):
            sub = df_res[df_res.modelo == nome]; a_.scatter(x + rng.uniform(-0.08, 0.08, len(sub)), sub[m], s=45, alpha=.8, label=None); a_.hlines(sub[m].mean(), x - 0.25, x + 0.25, color="k", lw=1.5)
            o = sub[sub.original]; a_.scatter(x, o[m], s=170, facecolors="none", edgecolors="r", lw=1.5, label="semente original" if x == 0 else None)
        a_.set_xticks(range(len(BASE))); a_.set_xticklabels(list(BASE)); a_.set_title(tit); a_.grid(axis="y", alpha=.3)
    ax[0].legend(fontsize=8); fig.suptitle(f"Variação entre {len(SEMENTES)} sementes (cada ponto = 1 semente; traço = média; círculo vermelho = semente das etapas anteriores)"); fig.tight_layout(); fig.savefig(PASTA_EST / "grafico_sementes.png", dpi=130); plt.close(fig); print("\n✔ gráfico salvo: grafico_sementes.png")
except ImportError:
    print("\n(matplotlib não está instalado: gráfico não gerado)")

== Resultados no teste, por modelo e semente (original = a semente das etapas anteriores) ==


,modelo,semente,original,roc_auc,pr_auc_benign,recall_05,fpr_05,mcc_05,fn_05,fp_05,recall_syn_05,recall_fpr1pct_teste,limiar_0_1pct_validacao,recall_lim,fpr_lim,mcc_lim,fn_lim,fp_lim,tempo_previsao_s
0,padrão,42,True,0.98993,0.97276,0.95111,0.02095,0.35521,350177.0,1178.0,0.92849,0.34927,0.46000,0.99923,0.02654,0.94001,5502.0,1492.0,18.94247
1,padrão,43,False,0.98943,0.97006,0.94882,0.02293,0.34708,366631.0,1289.0,0.92510,0.36507,0.46000,0.99894,0.02999,0.92214,7589.0,1686.0,21.26519
2,padrão,44,False,0.99120,0.97790,0.95084,0.01870,0.35512,352131.0,1051.0,0.92808,0.59254,0.46000,0.99963,0.02151,0.96612,2626.0,1209.0,20.47900
3,padrão,45,False,0.98865,0.96551,0.94741,0.02433,0.34225,376699.0,1368.0,0.92307,0.38470,0.46000,0.95211,0.02988,0.35537,343008.0,1680.0,20.31935
4,padrão,46,False,0.99147,0.98101,0.99865,0.01781,0.91365,9652.0,1001.0,0.99817,0.41709,0.46000,0.99938,0.02259,0.95054,4444.0,1270.0,20.22747
5,ajustado,42,True,0.99136,0.97473,0.95259,0.01763,0.36144,339595.0,991.0,0.93065,0.56489,0.41599,0.99972,0.03017,0.96682,2016.0,1696.0,45.01805
6,ajustado,43,False,0.99145,0.97944,0.95277,0.01822,0.36186,338295.0,1024.0,0.93092,0.57460,0.41599,0.99979,0.02834,0.97212,1515.0,1593.0,43.98226
7,ajustado,44,False,0.99141,0.97692,0.95143,0.01713,0.35764,347928.0,963.0,0.92895,0.60790,0.41599,0.99951,0.02971,0.95450,3502.0,1670.0,43.59227
8,ajustado,45,False,0.99091,0.97733,0.95182,0.01709,0.35900,345083.0,961.0,0.92953,0.38892,0.41599,0.99960,0.02540,0.96230,2838.0,1428.0,43.96002
9,ajustado,46,False,0.99076,0.97221,0.95148,0.01902,0.35714,347558.0,1069.0,0.92902,0.40990,0.41599,0.99970,0.03040,0.96526,2184.0,1709.0,44.24941



== Resumo por modelo (média, desvio entre sementes, mínimo e máximo) ==


padrão                                              ajustado                                        
                              mean           std         min           max          mean         std           min           max
roc_auc                    0.99014       0.00119     0.98865       0.99147       0.99118     0.00032       0.99076       0.99145
pr_auc_benign              0.97345       0.00616     0.96551       0.98101       0.97613     0.00275       0.97221       0.97944
recall_05                  0.95937       0.02201     0.94741       0.99865       0.95202     0.00063       0.95143       0.95277
fpr_05                     0.02094       0.00275     0.01781       0.02433       0.01782     0.00081       0.01709       0.01902
mcc_05                     0.46266       0.25217     0.34225       0.91365       0.35942     0.00216       0.35714       0.36186
fn_05                 291058.00000  157686.36873  9652.00000  376699.00000  343691.80000  4492.70383  338295.00000  347928.00000
fp_05                   1177.40000     154.82022  1001.00000    1368.00000    1001.60000    45.53899     961.00000    1069.00000
recall_syn_05              0.94058       0.03227     0.92307       0.99817       0.92982     0.00092       0.92895       0.93092
recall_fpr1pct_teste       0.42173       0.09878     0.34927       0.59254       0.50924     0.10179       0.38892       0.60790
mcc_lim                    0.82684       0.26404     0.35537       0.96612       0.96420     0.00649       0.95450       0.97212
fn_lim                 72633.80000  151154.43550  2626.00000  343008.00000    2411.00000   771.84519    1515.00000    3502.00000
fp_lim                  1467.40000     223.23037  1209.00000    1686.00000    1619.20000   115.95991    1428.00000    1709.00000


== Diferença pareada por semente (ajustado − padrão) ==


,métrica,melhor é,diferença média (ajustado − padrão),desvio das diferenças,mínima,máxima,ajustado melhor em,na semente original
0,roc_auc,maior,0.00104,0.00126,-0.00071,0.00226,4 de 5,0.00143
1,pr_auc_benign,maior,0.00268,0.00828,-0.00880,0.01182,3 de 5,0.00197
2,recall_05,maior,-0.00735,0.02232,-0.04717,0.00441,4 de 5,0.00148
3,fpr_05,menor,-0.00313,0.00319,-0.00724,0.00121,4 de 5,-0.00333
4,mcc_05,maior,-0.10325,0.25345,-0.55652,0.01675,4 de 5,0.00623
5,fn_05,menor,52633.80000,159891.38861,-31616.00000,337906.00000,4 de 5,-10582.00000
6,fp_05,menor,-175.80000,179.36192,-407.00000,68.00000,4 de 5,-187.00000
7,recall_syn_05,maior,-0.01077,0.03272,-0.06915,0.00647,4 de 5,0.00216
8,recall_fpr1pct_teste,maior,0.08751,0.11447,-0.00718,0.21562,4 de 5,0.21562
9,mcc_lim,maior,0.13736,0.26344,-0.01162,0.60694,4 de 5,0.02681



ROC-AUC no teste, 5 sementes: padrão 0.98865 a 0.99147 (média 0.99014, desvio 0.00119) | ajustado 0.99076 a 0.99145 (média 0.99118, desvio 0.00032)
→ as faixas SE SOBREPÕEM: em alguma semente o padrão superou o pior ajustado (a diferença não é maior que o ruído do sorteio).
Diferença pareada: média +0.00104 (de -0.00071 a +0.00226); na semente original (42), a das etapas anteriores: +0.00143; o ajustado foi melhor em 4 de 5 sementes.
Recall do `Syn` no limiar 0,5: padrão 0.9231 a 0.9982 | ajustado 0.9289 a 0.9309

✔ gráfico salvo: grafico_sementes.png


## 5. Gravação e conferências
Grava os resultados na pasta `estudo_sementes` e confere que a semente original reproduz **exatamente** os números do notebook 08 (isso prova que a leitura, os modelos e as métricas são os mesmos). Nenhum arquivo do notebook 08 é alterado.

In [12]:
# ---------- 5.1 Gravação e conferências ----------
df_res.to_csv(PASTA_EST / "resultados_por_semente.csv", index=False); resumo_tab.to_csv(PASTA_EST / "resumo_por_modelo.csv"); tab_par.to_csv(PASTA_EST / "diferenca_pareada.csv", index=False)
salvar_json({"data": datetime.now().isoformat(timespec="seconds"), "run_id_ajustado": PASTA_RUN.name, "modelo_padrao": PASTA_PADRAO.name, "sementes": SEMENTES, "semente_original": SEMENTE_ORIG, "n_teste": len(Y), "lote_linhas": LOTE_LINHAS,
             "hiperparametros": {n: {k: BASE[n][1][k] for k in ("n_estimators", "max_depth", "min_samples_leaf")} for n in BASE}, "limiar_0_1pct_validacao": LIM_01,
             "resultados": df_res.to_dict(orient="records"), "diferenca_pareada": tab_par.to_dict(orient="records"), "reproduz_modelo_ajustado_original": REPRODUZ,
             "observacao": "estudo DESCRITIVO da variação por semente; usa o teste de novo; nenhuma escolha depende dele; não mede a variação por outros dados de treino"}, PASTA_EST / "estudo_sementes.json")
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Previsões = linhas do teste", len(Y) == N_TESTE, f"({len(Y):,})")
verifica("Probabilidades entre 0 e 1 em todos os modelos", all(bool(((p >= 0) & (p <= 1)).all()) for p in PS), f"({len(PS)} modelos)")
verifica("Lotes gravados somam o teste", sum(n_lote) == N_TESTE and len(list(PASTA_LOTES.glob(f"lote_*_de_{B:03d}.npz"))) == B, f"({B} lotes)")
verifica("Número de modelos = 2 × sementes", len(CHAVES) == 2 * len(SEMENTES) == len(df_res))
verifica("Sementes diferentes dão modelos diferentes (o sorteio faz efeito)", all(df_res[df_res.modelo == n].roc_auc.nunique() > 1 for n in BASE) if len(SEMENTES) > 1 else True)
ref_json = PASTA_RUN / "metricas_teste.json"
if ref_json.exists():
    ref = json.loads(ref_json.read_text(encoding="utf-8")); r_aj, r_pad = ref["metricas"]["limiar 0.5"], (ref.get("metricas_padrao") or {}).get("limiar 0.5")
    o = df_res[df_res.original].set_index("modelo")
    verifica("A semente original reproduz o ROC-AUC e os erros do notebook 08 (ajustado)", abs(o.loc["ajustado", "roc_auc"] - r_aj["roc_auc"]) < 1e-6 and int(o.loc["ajustado", "fn_05"]) == int(r_aj["fn"]) and int(o.loc["ajustado", "fp_05"]) == int(r_aj["fp"]), f"(ROC-AUC {o.loc['ajustado', 'roc_auc']:.5f})")
    if r_pad: verifica("A semente original reproduz o ROC-AUC e os erros do notebook 08 (padrão)", abs(o.loc["padrão", "roc_auc"] - r_pad["roc_auc"]) < 1e-6 and int(o.loc["padrão", "fn_05"]) == int(r_pad["fn"]) and int(o.loc["padrão", "fp_05"]) == int(r_pad["fp"]), f"(ROC-AUC {o.loc['padrão', 'roc_auc']:.5f})")
else:
    print("(metricas_teste.json do notebook 08 não encontrado: conferência com o 08 omitida)")
if REPRODUZ is not None: verifica("O ajustado retreinado com a semente original é idêntico ao do notebook 07", REPRODUZ)
verifica("Arquivos gravados", all((PASTA_EST / f).exists() for f in ("resultados_por_semente.csv", "resumo_por_modelo.csv", "diferenca_pareada.csv", "estudo_sementes.json")))
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_EST}")

✔ Previsões = linhas do teste (7,219,189)
✔ Probabilidades entre 0 e 1 em todos os modelos (10 modelos)
✔ Lotes gravados somam o teste (145 lotes)
✔ Número de modelos = 2 × sementes 
✔ Sementes diferentes dão modelos diferentes (o sorteio faz efeito) 
✔ A semente original reproduz o ROC-AUC e os erros do notebook 08 (ajustado) (ROC-AUC 0.99136)
✔ A semente original reproduz o ROC-AUC e os erros do notebook 08 (padrão) (ROC-AUC 0.98993)
✔ O ajustado retreinado com a semente original é idêntico ao do notebook 07 
✔ Arquivos gravados 

✔ 9/9 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_binario_busca\rf_busca_20261024_175105\estudo_sementes
